# Garimpo measurement

How well do the project's data measure garimpo? This notebook runs the measurement
checks recorded as Evidence E3, E5 and E6 in
[`output/research/mining_economics_leads.md`](../../research/mining_economics_leads.md):

1. **E3, the MapBiomas Collection 11 mining-substances layer**: build checks, the national
   garimpo-gold series and its 2020 step, and how many municipalities have garimpo gold
   within 50 km upstream;
2. **E5, SIVEP-Malária as a measure of garimpo workers**: the sample by year, the
   local/incoming split, infection abroad, who the local cases are, whether counts follow
   testing, and the fit to mapped mining (C10, and C11 garimpo gold and garimpo of every
   substance, local and upstream);
3. **E6, SGB gold geology**: its fit to C11 garimpo gold and its relevance for garimpo
   cases.

Inputs are the processed tables (`land_cover_mining*`, `malaria_adm2`,
`malaria_garimpo_flows`, `geology_adm2`, `adm2_panel`); the raw SIVEP files are read
only for the demographics, the notification site and foreign residents, with the
columns restricted. Tables are written to `output/analysis/garimpo/measurement/`.

The Legal Amazon is taken as its nine states in full (808 municipalities, all of
Maranhão).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from scipy.spatial import cKDTree

from src.analysis import health as H
from src.analysis import panels
from src.data.sources.geology.constants import gpkg_path
from src.data.sources.geology.preprocess import load_gold_occurrences
from src.data.sources.land_cover.mining import (
    MINING_DATADIR, MINING_EXTRACT_PATH, MINING_FILENAME_TEMPLATE, MINING_GROUPS, MINING_NATIONAL_AREA_PATH,
    mining_group,
)
from src.data.sources.malaria.constants import READ_COLUMNS, YEARS, year_path
from src.data.sources.malaria.preprocess import classify_records

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:.4g}")

OUT = PROJECT_ROOT / "output" / "analysis" / "garimpo" / "measurement"
OUT.mkdir(parents=True, exist_ok=True)
UF = {"11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO", "21": "MA", "22": "PI",
      "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA", "31": "MG", "32": "ES",
      "33": "RJ", "35": "SP", "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF"}
LEGAL_AMAZON = {"11", "12", "13", "14", "15", "16", "17", "21", "51"}
GROUPS = list(MINING_GROUPS.values())

gadm = gpd.read_file(PROJECT_ROOT / "data/gadm/processed/gadm_simplified.gpkg", layer="ADM_ADM_2", ignore_geometry=True)
NAMES = gadm.assign(mun_id=gadm["CC_2"].astype(str).str[:6]).drop_duplicates("mun_id").set_index("mun_id")["NAME_2"]


def label(mun_id):
    return f"{NAMES.get(mun_id, '?')} ({UF.get(str(mun_id)[:2], '?')})" if pd.notna(mun_id) else "abroad"


panel = panels.load_adm2_panel(years=None)
municipalities = panel.drop_duplicates("mun_id").set_index("mun_id")
legal_amazon = municipalities.index[municipalities.index.str[:2].isin(LEGAL_AMAZON)]
print(f"adm2_panel: {len(municipalities):,} municipalities, {len(legal_amazon)} in the Legal Amazon states")

adm2_panel: 5,564 municipalities, 808 in the Legal Amazon states


## 1. E3: the C11 mining-substances layer

### 1.1 Build checks

The extract must keep each trench's pixel total constant across years (the groups and
phases adding up is checked by `data verify --source land_cover`). Its mined pixels are
compared with the national table written when the Earth Engine tiles were merged.

In [2]:
extract = pd.read_parquet(PROJECT_ROOT / MINING_EXTRACT_PATH,
                          columns=["trench_id", "year", "land_cover_total", *[f"land_cover_class_{g}" for g in MINING_GROUPS]])
varying = int(extract.groupby("trench_id")["land_cover_total"].nunique().gt(1).sum())
print(f"{extract.trench_id.nunique():,} trenches × {extract.year.nunique()} years; trenches whose total varies: {varying}")

area = pd.read_csv(PROJECT_ROOT / MINING_NATIONAL_AREA_PATH)
area["group"] = area["class"].map(mining_group).map(MINING_GROUPS)
national_pixels = area.groupby("year")["pixels"].sum()
extract_pixels = extract.groupby("year")[[f"land_cover_class_{g}" for g in MINING_GROUPS]].sum().sum(axis=1)
ratio = (extract_pixels / national_pixels).rename("extract / national mined pixels")
print(f"extract / national mined pixels: {ratio.min():.4f}–{ratio.max():.4f}")
ratio.to_frame().T.round(4)

404,002 trenches × 41 years; trenches whose total varies: 0


extract / national mined pixels: 1.0134–1.0178


year,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
extract / national mined pixels,1.018,1.018,1.017,1.016,1.015,1.015,1.014,1.014,1.014,1.014,1.014,1.014,1.014,1.014,1.014,1.014,1.014,1.015,1.015,1.015,...,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.015,1.014,1.014,1.014,1.014,1.014,1.014,1.013


**Reading the results.** The extract is internally consistent: no trench's pixel total
varies across years. It holds 1.3–1.8% more mined pixels than the national table, a
constant ratio, which fits trenches that straddle the border or the coast being counted
in full; the series it gives is the national series up to that scale.

### 1.2 National series and the 2020 step

In [3]:
km2 = area.pivot_table(index="year", columns="group", values="area_km2", aggfunc="sum").reindex(columns=GROUPS).fillna(0)
km2["all mining"] = km2[GROUPS].sum(axis=1)
km2.to_csv(OUT / "c11_national_km2.csv")
growth = 100 * km2.pct_change()
display(km2.loc[[1985, 1990, 2000, 2005, 2010, 2013, 2015, 2019, 2020, 2021, 2025]].round(0))
display(growth.loc[2012:2025].round(1).rename_axis("growth (%)"))

by_class = area.pivot_table(index="year", columns="class", values="area_km2").fillna(0)
change = by_class.diff().iloc[1:]
print(f"class-years that fall: {int((change < 0).sum().sum())} of {change.size}")
print("garimpo gold growth (%) by period: ",
      {period: round(float(growth.loc[a:b, "garimpo_gold"].mean()), 1) for period, (a, b) in
       {"1986-90": (1986, 1990), "2000-11": (2000, 2011), "2012-15": (2012, 2015), "2016-19": (2016, 2019)}.items()})

group,garimpo_gold,garimpo_unknown,garimpo_other,industrial_gold,industrial_other,all mining
year,,,,,,
1985,202,25,93,28,392,740
1990,878,115,246,60,733,2033
2000,1251,166,315,98,988,2818
2005,1292,179,332,106,1112,3022
2010,1348,194,379,129,1318,3367
2013,1440,207,413,146,1464,3670
2015,1524,218,442,154,1548,3885
2019,1804,302,517,168,1680,4471
2020,2076,386,593,190,1813,5058


group,garimpo_gold,garimpo_unknown,garimpo_other,industrial_gold,industrial_other,all mining
growth (%),,,,,,
2012,2.3,2.4,2.8,4.4,3.6,3
2013,2.6,2.8,3.8,4.4,3.4,3.1
2014,2.7,2.8,3.2,2.6,2.9,2.8
2015,3.1,2.6,3.6,2.7,2.8,3
2016,4,5.8,4.3,2.9,2.3,3.4
2017,4.4,9,4.6,2.2,2.1,3.7
2018,4.3,8.2,3.2,1.7,1.8,3.3
2019,4.6,11,3.9,2.2,2.2,3.9
2020,15.1,27.7,14.7,12.8,7.9,13.1


class-years that fall: 870 of 3640
garimpo gold growth (%) by period:  {'1986-90': 34.5, '2000-11': 0.8, '2012-15': 2.7, '2016-19': 4.3}


**Reading the results.** Garimpo gold grows from about 1,250 km² in 2000 to 1,800 km²
in 2019 and 2,900 km² in 2025. Growth is under 1% a year in 2000–11, 2.7% in 2012–15 and
4.3% in 2016–19, then 15% in 2020. The 2020 step appears in every class, including
industrial mining (13% gold, 8% other), so part of it is a change in the classification
between years rather than garimpo alone; garimpo of unknown substance grows fastest. A
quarter of the yearly changes in the national area of a legend class are declines, so the
layer is not a cumulative scar map.

### 1.3 Exposure within 50 km upstream

On the health estimation frame (2000–2024): how many municipalities have garimpo gold
within 50 km upstream, where, and how often it switches on (gold present, the two
previous years observed without gold).

In [4]:
frame = H.build_health_panel()[["mun_id", "year", "biome", "c11_garimpo_gold_50"]]
frame["gold"] = frame["c11_garimpo_gold_50"].fillna(0) > 0
frame["state"] = frame["mun_id"].str[:2].map(UF)
by_year = frame[frame.gold].groupby("year")["mun_id"].nunique()
ever = frame.loc[frame.gold, ["mun_id", "state"]].drop_duplicates()
print(f"municipalities with gold within 50 km: {by_year.min()}–{by_year.max()} a year, {len(ever)} ever;",
      ever["state"].value_counts().head(6).to_dict())
unlimited = panels.adm2_upstream_mining(years=H.ANALYSIS_YEARS)
unlimited = unlimited[unlimited["mun_id"].isin(set(frame["mun_id"])) & (unlimited["mining_garimpo_gold"] > 0)]
print(f"without the 50 km cap: {unlimited.mun_id.nunique()} municipalities ever")

frame = frame.sort_values(["mun_id", "year"]).reset_index(drop=True)
by_mun = frame.groupby("mun_id")
frame["onset"] = (frame.gold & by_mun.gold.shift(1).eq(False) & by_mun.gold.shift(2).eq(False)
                  & by_mun.year.shift(1).eq(frame.year - 1) & by_mun.year.shift(2).eq(frame.year - 2))
onsets = frame[frame.onset]
print(f"onsets: {len(onsets)} in {onsets.mun_id.nunique()} municipalities;",
      onsets["state"].value_counts().to_dict(), f"| Amazon biome: {(onsets.biome == 'Amazônia').sum()}",
      f"| median share at onset: {onsets.c11_garimpo_gold_50.median():.2g}")
by_year.rename("municipalities with gold within 50 km").to_frame().T

municipalities with gold within 50 km: 112–134 a year, 137 ever; {'MG': 51, 'PA': 27, 'MT': 22, 'MA': 10, 'AP': 7, 'RO': 5}


without the 50 km cap: 678 municipalities ever
onsets: 39 in 27 municipalities; {'MG': 34, 'AP': 3, 'PA': 1, 'MA': 1} | Amazon biome: 5 | median share at onset: 1.3e-06


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
municipalities with gold within 50 km,112,115,118,121,121,121,118,120,117,114,114,120,125,126,126,123,123,126,127,130,124,126,126,131,134


**Reading the results.** Within 50 km upstream, garimpo gold reaches 112–134
municipalities a year (137 ever), most in Minas Gerais; without the reach cap, 678.
Switch-ons are rare and tiny: 39 onsets in 27 municipalities, 34 of them in Minas
Gerais, only 5 in the Amazon biome, with a median share at onset of about one pixel in a
million. The upstream gold measure therefore identifies through intensity changes in a
fixed set of exposed municipalities, not through new exposure.

## 2. E5: SIVEP-Malária as a measure of garimpo workers

### 2.1 One pass over the raw files

`classify_records` applies the sample rules of `src.data.sources.malaria` (positives,
cure-verification slides dropped, garimpo = occupation "Garimpagem", placeholder codes
as missing). On top of the aggregated tables, the pass keeps every garimpo case and the
other cases resident in the 25 largest garimpo infection municipalities, with sex, age,
schooling and race, and counts exams and positives by notification municipality.

In [5]:
malaria = pd.read_parquet(PROJECT_ROOT / "data/malaria/processed/malaria_adm2.parquet")
flows = pd.read_parquet(PROJECT_ROOT / "data/malaria/processed/malaria_garimpo_flows.parquet")
TOP25 = set(malaria.groupby("mun_id")["inf_garimpo"].sum().nlargest(25).index)
DEMOGRAPHICS = ["sexo", "id_pacie", "id_dimea", "niv_esco", "raca"]
AGE_UNIT = {"Anos": 1.0, "Meses": 1 / 12, "Dias": 1 / 365}

samples, cases_kept, notification = [], [], []
for year in YEARS:
    raw = pd.read_parquet(year_path(PROJECT_ROOT, year), columns=[*READ_COLUMNS, *DEMOGRAPHICS])
    records = classify_records(raw)
    for column in DEMOGRAPHICS:
        records[column] = raw[column].to_numpy()
    garimpo = records["case"] & records["garimpo"]
    brazil = garimpo & ~records["abroad"]
    samples.append({
        "year": year, "positives": int(records["case"].sum()), "garimpo": int(garimpo.sum()),
        "garimpo infected abroad": int((garimpo & records["abroad"]).sum()),
        "garimpo infected in RR": int((brazil & records["inf"].str[:2].eq("14")).sum()),
        "Brazil-infected garimpo": int(brazil.sum()),
        "valid infection municipality": int((brazil & records["inf"].notna()).sum()),
        "valid residence municipality": int((brazil & records["res"].notna()).sum()),
    })
    keep = garimpo | (records["case"] & records["res"].isin(TOP25))
    cases_kept.append(records[keep])
    notification.append(records[records["noti"].notna()].assign(
        garimpo_positive=lambda r: r["case"] & r["garimpo"], other_positive=lambda r: r["case"] & ~r["garimpo"],
        negative=lambda r: r["exam"] & ~r["case"],
    ).groupby(["noti", "year"])[["exam", "case", "garimpo_positive", "other_positive", "negative"]].sum())
    del raw, records

samples = pd.DataFrame(samples).set_index("year")
cases = pd.concat(cases_kept, ignore_index=True)
cases["age"] = pd.to_numeric(cases["id_pacie"], errors="coerce") * cases["id_dimea"].map(AGE_UNIT)
notification = pd.concat(notification).reset_index()
samples.to_csv(OUT / "sivep_sample_by_year.csv")
print(f"{samples.positives.sum():,} positives, {samples.garimpo.sum():,} garimpo cases, 2004–2024")

5,379,472 positives, 312,276 garimpo cases, 2004–2024


### 2.2 Sample by year and completeness

In [6]:
table = samples.assign(**{"garimpo share (%)": 100 * samples.garimpo / samples.positives})
display(table[["positives", "garimpo", "garimpo share (%)", "garimpo infected in RR", "garimpo infected abroad"]])
print("Brazil-infected garimpo cases with a valid infection / residence municipality: "
      f"{samples['valid infection municipality'].sum() / samples['Brazil-infected garimpo'].sum():.4%} / "
      f"{samples['valid residence municipality'].sum() / samples['Brazil-infected garimpo'].sum():.4%}")

,positives,garimpo,garimpo share (%),garimpo infected in RR,garimpo infected abroad
year,,,,,
2004,464899,22684,4.879,52,6357
2005,606068,19393,3.2,27,4279
2006,549380,17667,3.216,12,5139
2007,457469,17587,3.844,18,4850
2008,314906,14562,4.624,17,3168
2009,308400,16537,5.362,24,3144
2010,333461,18764,5.627,40,4469
2011,265848,13785,5.185,50,3654
2012,241806,14542,6.014,36,3428


Brazil-infected garimpo cases with a valid infection / residence municipality: 99.9940% / 99.9835%


**Reading the results.** About 312,000 garimpo cases in 21 years. Their share of
positives rises from 3–6% before 2019 to 11–18% in 2020–24, while all positives fall by
two thirds. Roraima's garimpo cases jump from tens a year to 12,000 in 2022 (the
Yanomami territory), and infection abroad falls from several thousand a year to under
1,300 after 2019. Infection and residence municipalities are almost always valid, so the
municipality assignment is not a source of loss.

### 2.3 Local and incoming cases

In [7]:
local, incoming = malaria["inf_garimpo_local"].sum(), malaria["inf_garimpo_incoming"].sum()
print(f"local {local / (local + incoming):.1%}, incoming {incoming / (local + incoming):.1%}, "
      f"from another state {malaria['inf_garimpo_incoming_other_state'].sum() / (local + incoming):.1%}")
yearly = malaria.groupby("year")[["inf_garimpo_local", "inf_garimpo_incoming"]].sum()
print("local share by year:", (yearly.inf_garimpo_local / yearly.sum(axis=1)).round(2).to_dict())

hubs = malaria.groupby("mun_id")[["inf_garimpo", "inf_garimpo_local", "inf_garimpo_incoming", "inf_garimpo_incoming_other_state"]].sum()
hubs = hubs.nlargest(12, "inf_garimpo")
hubs["local share"] = hubs.inf_garimpo_local / (hubs.inf_garimpo_local + hubs.inf_garimpo_incoming)
hubs.index = hubs.index.map(label)
display(hubs.round(3))

moving = flows[flows.inf_mun_id.notna() & (flows.res_mun_id != flows.inf_mun_id)]
top_flows = moving.groupby(["res_mun_id", "inf_mun_id"])["cases"].sum().nlargest(10).reset_index()
top_flows["residence"], top_flows["infection"] = top_flows.res_mun_id.map(label), top_flows.inf_mun_id.map(label)
top_flows[["residence", "infection", "cases"]]

local 60.3%, incoming 39.7%, from another state 13.5%
local share by year: {2004: 0.6, 2005: 0.66, 2006: 0.71, 2007: 0.72, 2008: 0.67, 2009: 0.72, 2010: 0.75, 2011: 0.73, 2012: 0.73, 2013: 0.76, 2014: 0.7, 2015: 0.68, 2016: 0.64, 2017: 0.58, 2018: 0.54, 2019: 0.52, 2020: 0.52, 2021: 0.44, 2022: 0.3, 2023: 0.48, 2024: 0.71}


,inf_garimpo,inf_garimpo_local,inf_garimpo_incoming,inf_garimpo_incoming_other_state,local share
mun_id,,,,,
Itaituba (PA),80329,62647,17679,7614,0.78
Jacareacanga (PA),33402,28673,4729,896,0.858
Alto Alegre (RR),22241,461,21777,1130,0.021
Calçoene (AP),12220,11257,963,161,0.921
Novo Progresso (PA),11498,9488,2009,702,0.825
Maués (AM),9054,1128,7926,7599,0.125
Mucajaí (RR),7697,545,7151,381,0.071
Itapuã do Oeste (RO),7270,2019,5251,38,0.278
Almeirim (PA),4474,253,4220,4119,0.057


,residence,infection,cases
0,Boa Vista (RR),Alto Alegre (RR),19676
1,Boa Vista (RR),Mucajaí (RR),6238
2,Novo Progresso (PA),Itaituba (PA),4189
3,Ariquemes (RO),Itapuã do Oeste (RO),4168
4,Laranjal do Jari (AP),Almeirim (PA),3986
5,Itaituba (PA),Maués (AM),3671
6,Jacareacanga (PA),Maués (AM),3423
7,Maués (AM),Itaituba (PA),3136
8,Itaituba (PA),Jacareacanga (PA),2890
9,Santarém (PA),Itaituba (PA),2565


**Reading the results.** Six in ten garimpo cases are infected in their municipality
of residence, falling from about 0.7 before 2016 to 0.3 in 2022, when the Roraima cases
are mostly residents of Boa Vista infected in Alto Alegre and Mucajaí. The largest
infection municipalities split into two kinds: Tapajós municipalities (Itaituba,
Jacareacanga, Novo Progresso) and Calçoene, where 78–92% of cases are local, and
municipalities supplied from elsewhere (Alto Alegre, Mucajaí, Maués, Almeirim), where
5–13% are. Cases by residence and by infection therefore measure different things in the
second group.

### 2.4 Infection abroad

In [8]:
garimpo_cases = cases[cases["case"] & cases["garimpo"]]
abroad = garimpo_cases[garimpo_cases["abroad"]]
print(f"infected abroad: {len(abroad):,} ({len(abroad) / len(garimpo_cases):.1%} of garimpo cases)")
display(abroad["country"].value_counts().head(6).rename("garimpo cases infected abroad").to_frame().T)
foreign = abroad[abroad["res"].isna()]
print(f"without a Brazilian residence: {len(foreign):,}; Venezuela-infected among them: "
      f"{(foreign.country == 'Venezuela').sum():,}")
foreign.groupby("year").size().rename("abroad, no Brazilian residence").to_frame().T

infected abroad: 64,092 (20.5% of garimpo cases)


country,Guiana Francesa,Venezuela,Guiana,Suriname,Franca,Bolivia
garimpo cases infected abroad,33594,17236,11358,1511,219,71


without a Brazilian residence: 5,391; Venezuela-infected among them: 4,933


year,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
"abroad, no Brazilian residence",192,106,63,63,31,32,276,73,95,255,137,270,1036,810,1738,95,34,12,32,20,21


**Reading the results.** One garimpo case in five is infected abroad, mostly in French
Guiana, Venezuela and Guyana. Almost all of these patients live in Brazil; the 5,391
without a Brazilian residence are mostly Venezuelans, concentrated in 2016–18.

### 2.5 Who the local cases are

2013–2024, when race and the current schooling coding are complete: local and incoming
garimpo cases (Brazil-infected, valid residence), and the other cases resident in the 25
largest garimpo infection municipalities.

In [9]:
SECONDARY_PLUS = {"Ensino medio completo", "Educacao superior incompleta", "Educacao superior completa"}


def profile(group):
    schooling = group["niv_esco"].dropna()
    return pd.Series({
        "n": len(group), "male": group["sexo"].eq("Masculino").mean(), "under 15": group["age"].lt(15).mean(),
        "median age": group["age"].median(), "indigenous": group["raca"].eq("Indigena").mean(),
        "secondary or more": schooling.isin(SECONDARY_PLUS).mean(), "schooling recorded": group["niv_esco"].notna().mean(),
    })


recent = cases[cases["year"].between(2013, 2024)]
recent_garimpo = recent[recent["garimpo"] & recent["inf"].notna() & recent["res"].notna()]
groups = {
    "garimpo, local": recent_garimpo[recent_garimpo["inf"] == recent_garimpo["res"]],
    "garimpo, incoming": recent_garimpo[recent_garimpo["inf"] != recent_garimpo["res"]],
    "other malaria, resident in top 25": recent[~recent["garimpo"] & recent["res"].isin(TOP25)],
}
demographics = pd.DataFrame({name: profile(group) for name, group in groups.items()}).T
demographics.to_csv(OUT / "sivep_demographics_2013_2024.csv")
display(demographics)

brazil_garimpo = garimpo_cases[garimpo_cases["inf"].notna() & garimpo_cases["res"].notna()]
incoming_cases = brazil_garimpo[brazil_garimpo["inf"] != brazil_garimpo["res"]]
local_cases = brazil_garimpo[brazil_garimpo["inf"] == brazil_garimpo["res"]]
print(f"incoming cases notified at their residence: {(incoming_cases.noti == incoming_cases.res).mean():.1%}; "
      f"local cases notified where infected: {(local_cases.noti == local_cases.inf).mean():.1%}")

,n,male,under 15,median age,indigenous,secondary or more,schooling recorded
"garimpo, local",6.849e+04,0.8409,0.04113,32,0.05698,0.1234,0.988
"garimpo, incoming",6.263e+04,0.7941,0.01375,32,0.008862,0.272,0.9798
"other malaria, resident in top 25",2.865e+05,0.5737,0.3576,21,0.4588,0.1264,0.9228


incoming cases notified at their residence: 85.1%; local cases notified where infected: 91.2%


**Reading the results.** Local and incoming garimpo cases look alike and unlike other
malaria in the same municipalities: about 80% men, median age 32, few children, against
57% men, median age 21 and a third under 15 for other cases. Local cases are more often
indigenous (6% vs. 1%) and less schooled (12% vs. 27% with secondary or more), so local
garimpo workers are drawn from the regional population, not only from migrants. Cases are
notified where the patient is: 85% of incoming cases at their residence, 91% of local
cases where infected.

### 2.6 Do garimpo counts follow testing?

Year-on-year changes in log(1 + count) by municipality, 2004–2024: garimpo positives
(by infection municipality from `malaria_adm2`, by notification municipality from the
raw pass) against exams, and other positives against negatives, all at the notification
municipality. If garimpo counts were driven by testing, they would move with exams as
closely as other malaria moves with negatives.

In [10]:
by_notification = notification.rename(columns={"noti": "mun_id"}).set_index(["mun_id", "year"])
by_infection = malaria.set_index(["mun_id", "year"])["inf_garimpo"].rename("garimpo_infection")
garimpo_totals = malaria.groupby("mun_id")["inf_garimpo"].sum()
sets = {
    "top 25 infection municipalities": TOP25,
    ">= 100 garimpo cases": set(garimpo_totals.index[garimpo_totals >= 100]),
}
pairs = [("garimpo_infection", "exam"), ("garimpo_positive", "exam"), ("garimpo_positive", "negative"),
         ("other_positive", "negative"), ("case", "negative")]
rows = []
for name, members in sets.items():
    index = pd.MultiIndex.from_product([sorted(members), list(YEARS)], names=["mun_id", "year"])
    counts = by_notification.reindex(index).join(by_infection).fillna(0)
    changes = np.log1p(counts).groupby(level="mun_id").diff().dropna()
    for x, y in pairs:
        within = changes.groupby(level="mun_id").apply(lambda g: g[x].corr(g[y]))
        rows.append({"set": name, "municipalities": len(members), "x": x, "y": y,
                     "pooled corr": changes[x].corr(changes[y]), "median within-municipality corr": within.median()})
testing = pd.DataFrame(rows)
testing.to_csv(OUT / "sivep_testing_covariation.csv", index=False)
testing

,set,municipalities,x,y,pooled corr,median within-municipality corr
0,top 25 infection municipalities,25,garimpo_infection,exam,0.1715,0.2769
1,top 25 infection municipalities,25,garimpo_positive,exam,0.2802,0.2406
2,top 25 infection municipalities,25,garimpo_positive,negative,0.1909,0.14
3,top 25 infection municipalities,25,other_positive,negative,0.4692,0.5849
4,top 25 infection municipalities,25,case,negative,0.4966,0.5715
5,>= 100 garimpo cases,61,garimpo_infection,exam,0.2143,0.2651
6,>= 100 garimpo cases,61,garimpo_positive,exam,0.2504,0.2255
7,>= 100 garimpo cases,61,garimpo_positive,negative,0.1796,0.1472
8,>= 100 garimpo cases,61,other_positive,negative,0.501,0.5849
9,>= 100 garimpo cases,61,case,negative,0.532,0.5703


**Reading the results.** Garimpo counts are only loosely tied to testing: their
year-on-year changes correlate with exams at 0.17–0.28, against 0.47–0.53 between other
positives and negatives. Testing intensity does not drive the garimpo series the way it
drives the rest of the malaria series.

### 2.7 Fit to mapped mining

Spearman correlation across the 808 Legal Amazon municipalities between 2010–2022
garimpo cases by infection municipality and the municipality's mean mined share over the
same years: C10 mining (local and upstream within 50 km), and C11 garimpo gold and
garimpo of every substance (gold, unknown, other), in the municipality's own trenches
and upstream within 50 and 200 km. Also: the share of case municipalities with none of
that mining, and the share of cases they hold.

In [11]:
window = panel[panel["year"].between(2010, 2022) & panel["mun_id"].isin(legal_amazon)]
c10_near = panels.adm2_upstream_land_cover(max_km=50, years=(2010, 2022))[["mun_id", "year", "lc_mining"]]
window = window.merge(c10_near.rename(columns={"lc_mining": "c10_upstream_50"}), on=["mun_id", "year"], how="left")
for scope, suffix in [("local", ""), ("upstream_50", "_50"), ("upstream_200", "_200")]:
    prefix = "local_c11_" if scope == "local" else "c11_"
    window[f"c11_gold_{scope}"] = window[f"{prefix}garimpo_gold{suffix}"]
    window[f"c11_garimpo_{scope}"] = window[[f"{prefix}garimpo_{group}{suffix}" for group in ("gold", "unknown", "other")]].sum(axis=1, min_count=1)
MEASURES = {
    "C10 mining, local": "local_lc_mining",
    "C10 mining, upstream 50 km": "c10_upstream_50",
    "C11 garimpo gold, local": "c11_gold_local",
    "C11 garimpo (all substances), local": "c11_garimpo_local",
    "C11 garimpo gold, upstream 50 km": "c11_gold_upstream_50",
    "C11 garimpo (all substances), upstream 50 km": "c11_garimpo_upstream_50",
    "C11 garimpo gold, upstream 200 km": "c11_gold_upstream_200",
    "C11 garimpo (all substances), upstream 200 km": "c11_garimpo_upstream_200",
}
means = window.groupby("mun_id")[list(MEASURES.values())].mean().fillna(0).reindex(legal_amazon).fillna(0)
garimpo_cases_window = malaria[malaria.year.between(2010, 2022)].groupby("mun_id")["inf_garimpo"].sum()
means["cases"] = garimpo_cases_window.reindex(means.index).fillna(0)
with_cases = means[means.cases > 0]
rows = []
for name, column in MEASURES.items():
    rows.append({
        "measure": name,
        "Spearman, 808 municipalities": means["cases"].corr(means[column], method="spearman"),
        "Spearman, municipalities with cases": with_cases["cases"].corr(with_cases[column], method="spearman"),
        "municipalities with the measure > 0": int((means[column] > 0).sum()),
        "case municipalities without it": (with_cases[column] == 0).mean(),
        "share of cases there": with_cases.loc[with_cases[column] == 0, "cases"].sum() / with_cases["cases"].sum(),
    })
fit = pd.DataFrame(rows)
fit.to_csv(OUT / "sivep_fit_to_mapped_mining.csv", index=False)
print(f"{len(with_cases)} municipalities with garimpo cases in 2010–2022, {int(means.cases.sum()):,} cases")
fit

337 municipalities with garimpo cases in 2010–2022, 133,855 cases


,measure,"Spearman, 808 municipalities","Spearman, municipalities with cases",municipalities with the measure > 0,case municipalities without it,share of cases there
0,"C10 mining, local",0.3739,0.3184,236,0.5401,0.0376
1,"C10 mining, upstream 50 km",0.2635,0.2134,206,0.632,0.1635
2,"C11 garimpo gold, local",0.3592,0.3176,80,0.7982,0.3253
3,"C11 garimpo (all substances), local",0.3645,0.3335,174,0.6409,0.09272
4,"C11 garimpo gold, upstream 50 km",0.2623,0.2371,75,0.8338,0.3936
5,"C11 garimpo (all substances), upstream 50 km",0.2863,0.2291,160,0.6884,0.1794
6,"C11 garimpo gold, upstream 200 km",0.2639,0.2511,131,0.7448,0.2172
7,"C11 garimpo (all substances), upstream 200 km",0.2962,0.238,254,0.5519,0.135


In [12]:
top = means.nlargest(15, "cases")[["cases", "local_lc_mining", "c11_gold_local", "c11_garimpo_local", "c11_gold_upstream_50", "c11_garimpo_upstream_50"]]
top.index = top.index.map(label)
display((top * [1, 100, 100, 100, 100, 100]).rename(columns=lambda c: c if c == "cases" else f"{c} (pp)"))

,cases,local_lc_mining (pp),c11_gold_local (pp),c11_garimpo_local (pp),c11_gold_upstream_50 (pp),c11_garimpo_upstream_50 (pp)
mun_id,,,,,,
Itaituba (PA),4.826e+04,0.3831,0.5074,0.6274,0.2826,0.3786
Alto Alegre (RR),1.83e+04,0.006112,0,0.01546,0,0.06338
Jacareacanga (PA),1.723e+04,0.1882,0.235,0.3252,0.4045,0.4192
Maués (AM),6720,0.02801,0.06027,0.06501,0.1727,0.1916
Calçoene (AP),5675,0.0348,0,0,0,0
Mucajaí (RR),4924,0.005647,0,0.01617,0,0.02932
Novo Progresso (PA),3292,0.1453,0.2137,0.2813,1.038,1.201
Pontes e Lacerda (MT),2629,0.06239,0.05769,0.07182,0,0
Pedra Branca do Amapari (AP),2624,0.05859,0,0.0005216,0,0


**Reading the results.** Every mapped measure ranks municipalities by garimpo cases
with a Spearman correlation of 0.26–0.37; the local measures fit best, and C11 gold does
no better than C10 mining (0.36 vs. 0.37 locally). C11 gold is the most restrictive: 80%
of the municipalities with cases have no local garimpo gold, and they hold a third of the
cases. Several large case municipalities have little or no mapped gold (Alto Alegre,
Calçoene, Mucajaí, Aripuanã), because their garimpos are in indigenous territories or
across the border, are mapped under another substance, or are too small for 30 m pixels.
Malaria and the C11 layer are complementary measures, not substitutes.

## 3. E6: SGB gold geology

### 3.1 Fit to C11 garimpo gold in 2020

Every garimpo-gold pixel of the 2020 C11 raster (codes 1115 and 1215), weighted by its
area: the share inside a gold province or district, and within 10 and 25 km of a gold
occurrence (all, or never exploited only).

In [13]:
raster_path = PROJECT_ROOT / MINING_DATADIR / MINING_FILENAME_TEMPLATE.format(year=2020)
lon, lat, weight = [], [], []
with rasterio.open(raster_path) as source:
    transform = source.transform
    for _, window_ in source.block_windows(1):
        block = source.read(1, window=window_)
        rows_, cols_ = np.nonzero((block // 1000 == 1) & (block % 100 == 15))
        if rows_.size == 0:
            continue
        x, y = rasterio.transform.xy(transform, rows_ + window_.row_off, cols_ + window_.col_off)
        lon.append(np.asarray(x)); lat.append(np.asarray(y))
lon, lat = np.concatenate(lon), np.concatenate(lat)
# Pixel centres; each pixel's area on the sphere, as `pixel_area_km2` computes it per row.
half = transform.a / 2
pixel_km2 = 6371.0072**2 * np.radians(transform.a) * (np.sin(np.radians(lat + half)) - np.sin(np.radians(lat - half)))
print(f"{lon.size:,} garimpo-gold pixels, {pixel_km2.sum():,.0f} km²")

pixels = gpd.GeoDataFrame({"km2": pixel_km2}, geometry=gpd.points_from_xy(lon, lat), crs=4326).to_crs(5880)
occurrences = load_gold_occurrences(gpkg_path(PROJECT_ROOT)).to_crs(5880)
shares = {}
for layer in ("gold_provinces", "gold_districts"):
    polygons = gpd.read_file(gpkg_path(PROJECT_ROOT), layer=layer).to_crs(5880)
    union = gpd.GeoDataFrame(geometry=[polygons.geometry.make_valid().union_all()], crs=5880)
    inside = gpd.sjoin(pixels, union, how="inner", predicate="within").index.unique()
    shares[f"inside a {layer.removeprefix('gold_')[:-1]}"] = pixels.loc[inside, "km2"].sum() / pixels["km2"].sum()
xy = np.column_stack([pixels.geometry.x, pixels.geometry.y])
for name, points in [("any occurrence", occurrences), ("a never-exploited occurrence", occurrences[occurrences.STATUS_ECONOMICO == "Não explotado"])]:
    distance, _ = cKDTree(np.column_stack([points.geometry.x, points.geometry.y])).query(xy)
    for km in (10, 25):
        shares[f"within {km} km of {name}"] = pixels.loc[distance <= km * 1000, "km2"].sum() / pixels["km2"].sum()
print(f"never-exploited occurrences: {(occurrences.STATUS_ECONOMICO == 'Não explotado').sum():,}")
pd.Series(shares, name="share of 2020 garimpo-gold area").to_frame().round(3)

2,342,313 garimpo-gold pixels, 2,076 km²


never-exploited occurrences: 1,073


,share of 2020 garimpo-gold area
inside a province,0.957
inside a district,0.443
within 10 km of any occurrence,0.922
within 25 km of any occurrence,0.984
within 10 km of a never-exploited occurrence,0.161
within 25 km of a never-exploited occurrence,0.477


**Reading the results.** The geology fits where garimpo is: 96% of the 2020 garimpo-gold
area lies inside a gold province and 92% within 10 km of a recorded occurrence. The
districts are narrower (44%). Never-exploited occurrences cover much less (16% within
10 km), so they are a weaker but cleaner instrument.

### 3.2 Relevance for garimpo cases

Spearman correlation across the 808 Legal Amazon municipalities between 2004–2024
garimpo cases by infection municipality and each geology measure.

In [14]:
geology = pd.read_parquet(PROJECT_ROOT / "data/geology/processed/geology_adm2.parquet").set_index("mun_id")
geology = geology.reindex(legal_amazon).fillna(0)
geology["cases"] = malaria.groupby("mun_id")["inf_garimpo"].sum().reindex(geology.index).fillna(0)
GEOLOGY = ["gold_occurrences", "gold_occurrences_unexploited", "gold_occurrences_garimpo", "gold_occurrences_mine",
           "gold_occurrences_alluvial", "gold_district_share", "gold_province_share"]
rows = []
for column in GEOLOGY:
    rows.append({"measure": column, "Spearman": geology["cases"].corr(geology[column], method="spearman"),
                 "municipalities > 0": int((geology[column] > 0).sum()),
                 "share of cases where 0": geology.loc[geology[column] == 0, "cases"].sum() / geology["cases"].sum()})
relevance = pd.DataFrame(rows)
relevance.to_csv(OUT / "geology_relevance.csv", index=False)
display(relevance)
largest = geology[geology.gold_occurrences_unexploited == 0].nlargest(6, "cases")
largest.index = largest.index.map(label)
display(largest[["cases", "gold_occurrences", "gold_district_share", "gold_province_share"]])
spots = geology.loc[["150360", "140002"], ["cases", "gold_occurrences_unexploited", "gold_district_share", "gold_province_share"]]
spots.index = spots.index.map(label)
display(spots)
rondonia = geology[geology.index.str[:2] == "11"].nlargest(5, "cases")
rondonia.index = rondonia.index.map(label)
display(rondonia[["cases", "gold_occurrences", "gold_province_share"]])
national = pd.read_parquet(PROJECT_ROOT / "data/geology/processed/geology_adm2.parquet").nlargest(10, "gold_occurrences")
print("top 10 municipalities by occurrences, by state:", national.mun_id.str[:2].map(UF).value_counts().to_dict())

,measure,Spearman,municipalities > 0,share of cases where 0
0,gold_occurrences,0.4063,154,0.06378
1,gold_occurrences_unexploited,0.3821,74,0.1467
2,gold_occurrences_garimpo,0.3422,89,0.1569
3,gold_occurrences_mine,0.1981,28,0.5575
4,gold_occurrences_alluvial,0.2945,51,0.5487
5,gold_district_share,0.3303,127,0.3071
6,gold_province_share,0.2933,140,0.189


,cases,gold_occurrences,gold_district_share,gold_province_share
mun_id,,,,
Maués (AM),9054,19,0,0.3741
Itapuã do Oeste (RO),7270,0,0,0
Porto Velho (RO),3992,32,0,0.02573
Ariquemes (RO),3274,2,0,0
Monte Negro (RO),2805,0,0,0
Campo Novo de Rondônia (RO),874,0,0,0


,cases,gold_occurrences_unexploited,gold_district_share,gold_province_share
mun_id,,,,
Itaituba (PA),8.033e+04,61,0.002337,0.6854
Amajari (RR),1609,71,0,0.9469


,cases,gold_occurrences,gold_province_share
mun_id,,,
Itapuã do Oeste (RO),7270,0,0
Espigão D'Oeste (RO),4065,4,0.7117
Porto Velho (RO),3992,32,0.02573
Ariquemes (RO),3274,2,0
Monte Negro (RO),2805,0,0


top 10 municipalities by occurrences, by state: {'MG': 4, 'PA': 3, 'RR': 1, 'MA': 1, 'AM': 1}


**Reading the results.** Occurrence counts rank municipalities by garimpo cases with a
Spearman correlation of 0.41, better than any land-cover measure, and municipalities with
no occurrence hold only 6% of the cases. The large case municipalities without
occurrences (Itapuã do Oeste, Ariquemes, Monte Negro) are in Rondônia's tin and
cassiterite belt, so their garimpo is likely not gold. Province shares and
never-exploited counts are weaker but still relevant (0.29–0.38).